<a href="https://colab.research.google.com/github/Aabhiinavvv/Hyperparameter-Tuning-Diabetecs-Prediction-Model-Using-Keras-Tuner/blob/main/diabetesKerasFineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd


In [5]:
df = pd.read_csv("/content/diabetes.csv")

In [6]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [7]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [8]:
x = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [9]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x = scaler.fit_transform(x)

In [10]:
x

array([[ 0.63994726,  0.84832379,  0.14964075, ...,  0.20401277,
         0.46849198,  1.4259954 ],
       [-0.84488505, -1.12339636, -0.16054575, ..., -0.68442195,
        -0.36506078, -0.19067191],
       [ 1.23388019,  1.94372388, -0.26394125, ..., -1.10325546,
         0.60439732, -0.10558415],
       ...,
       [ 0.3429808 ,  0.00330087,  0.14964075, ..., -0.73518964,
        -0.68519336, -0.27575966],
       [-0.84488505,  0.1597866 , -0.47073225, ..., -0.24020459,
        -0.37110101,  1.17073215],
       [-0.84488505, -0.8730192 ,  0.04624525, ..., -0.20212881,
        -0.47378505, -0.87137393]])

In [11]:
x.shape

(768, 8)

In [12]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=1 )

In [13]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense

In [14]:
model = Sequential()
model.add(Dense(32,activation='relu',input_dim=8))
model.add(Dense(1 ,activation='sigmoid'))
model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [15]:
model.fit(x_train , y_train,batch_size=32, epochs = 100,validation_data=(x_test,y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5961 - loss: 0.6814 - val_accuracy: 0.6948 - val_loss: 0.6488
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6466 - loss: 0.6253 - val_accuracy: 0.6818 - val_loss: 0.6012
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6775 - loss: 0.5870 - val_accuracy: 0.7078 - val_loss: 0.5722
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6906 - loss: 0.5609 - val_accuracy: 0.7273 - val_loss: 0.5485
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7117 - loss: 0.5415 - val_accuracy: 0.7532 - val_loss: 0.5315
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7199 - loss: 0.5254 - val_accuracy: 0.7597 - val_loss: 0.5171
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7378 - loss: 0.5124 - val_accuracy: 0.7857 - val_loss: 0.5050
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7606 - loss: 0.5011 - val_accuracy: 0.7922 - 

# 1)how to select appropriate optimizer
# 2)no,of nodes in a layer
# 3)how to select no of layers
#4)all in one model

In [16]:
pip install -U keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 6.7 MB/s eta 0:00:00


In [17]:
import kerastuner as kt

/tmp/ipykernel_3064/1654478174.py:1: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt


In [18]:
def build_model(hp):
    model = Sequential()
    model.add(Dense(32, activation="relu", input_dim=8))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=hp.Choice('optimizer', values=['adam','sgd','rmsprop','adadelta']),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [19]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials = 5)
tuner.results_summary()

Results summary
Results in ./untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")


In [20]:
tuner.search(x_train,y_train,epochs = 5,validation_data = (x_test,y_test))
tuner.results_summary()

Trial 4 Complete [00h 00m 03s]
val_accuracy: 0.7467532753944397

Best val_accuracy So Far: 0.7792207598686218
Total elapsed time: 00h 00m 19s
Results summary
Results in ./untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 2 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.7792207598686218

Trial 3 summary
Hyperparameters:
optimizer: adam
Score: 0.7467532753944397

Trial 1 summary
Hyperparameters:
optimizer: sgd
Score: 0.6948052048683167

Trial 0 summary
Hyperparameters:
optimizer: adadelta
Score: 0.5584415793418884


In [21]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [22]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [23]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
model.fit(x_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(x_test,y_test) )

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7622 - loss: 0.5238 - val_accuracy: 0.7922 - val_loss: 0.5093
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7671 - loss: 0.5074 - val_accuracy: 0.7857 - val_loss: 0.4969
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7687 - loss: 0.4974 - val_accuracy: 0.7792 - val_loss: 0.4895
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7671 - loss: 0.4895 - val_accuracy: 0.7727 - val_loss: 0.4845
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7801 - loss: 0.4836 - val_accuracy: 0.7857 - val_loss: 0.4798
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7801 - loss: 0.4782 - val_accuracy: 0.7857 - val_loss: 0.4758
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7818 - loss: 0.4738 - val_accuracy: 0.7857 - val_loss: 0.4727
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7785 - loss: 0.4700 - val_accuracy: 0.79

In [34]:
def build_model(hp):
  model = Sequential()
  units = hp.Int('units',min_value = 8,max_value=128,step = 8)
  model.add(Dense(units = units,activation='relu',input_dim = 8))
  model.add(Dense(1,activation='sigmoid'))
  model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics = ['accuracy'])
  return model


In [35]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials = 5,
    directory = 'mydir'
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [39]:
tuner.search(x_train,y_train,epochs=5,validation_data = (x_test,y_test))

Trial 5 Complete [00h 00m 02s]
val_accuracy: 0.7337662577629089

Best val_accuracy So Far: 0.7857142686843872
Total elapsed time: 00h 00m 12s


In [41]:
tuner.results_summary()

Results summary
Results in mydir/untitled_project
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 2 summary
Hyperparameters:
units: 104
Score: 0.7857142686843872

Trial 1 summary
Hyperparameters:
units: 56
Score: 0.7792207598686218

Trial 0 summary
Hyperparameters:
units: 120
Score: 0.7727272510528564

Trial 3 summary
Hyperparameters:
units: 48
Score: 0.7727272510528564

Trial 4 summary
Hyperparameters:
units: 16
Score: 0.7337662577629089


In [42]:
tuner.get_best_hyperparameters()[0].values

{'units': 104}

In [43]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [46]:
model.fit(x_train , y_train,batch_size=32, epochs = 100,initial_epoch=6,validation_data=(x_test,y_test) )

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8241 - loss: 0.4026 - val_accuracy: 0.8247 - val_loss: 0.4696
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8160 - loss: 0.4022 - val_accuracy: 0.8247 - val_loss: 0.4677
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8225 - loss: 0.4019 - val_accuracy: 0.8117 - val_loss: 0.4693
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8127 - loss: 0.4015 - val_accuracy: 0.8117 - val_loss: 0.4696
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8160 - loss: 0.4003 - val_accuracy: 0.7922 - val_loss: 0.4677
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8094 - loss: 0.4001 - val_accuracy: 0.8182 - val_loss: 0.4683
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8208 - loss: 0.3999 - val_accuracy: 0.8247 - val_loss: 0.4677
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8241 - loss: 0.3991 - val_accuracy: 0.81

In [52]:
def build_model(hp):
    model = Sequential()

    model.add(Dense(72, activation="relu", input_dim=8))

    num_layers = hp.Int('num_layers', min_value=1, max_value=10)

    for i in range(num_layers):
        model.add(Dense(72, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='rmsprop',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


In [53]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=3,
    directory='mydir',
    project_name="num_layers"
)


In [56]:
 tuner.search(x_train ,y_train,epochs = 5,validation_data =(x_test,y_test))

Trial 3 Complete [00h 00m 03s]
val_accuracy: 0.798701286315918

Best val_accuracy So Far: 0.798701286315918
Total elapsed time: 00h 00m 12s


In [59]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 4}

In [60]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [61]:
model.fit(x_train,y_train,batch_size = 32, epochs = 100,initial_epoch = 6, validation_data = (x_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.7834 - loss: 0.4588 - val_accuracy: 0.7922 - val_loss: 0.4769
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7834 - loss: 0.4414 - val_accuracy: 0.7922 - val_loss: 0.4772
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7818 - loss: 0.4297 - val_accuracy: 0.7922 - val_loss: 0.4652
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7964 - loss: 0.4259 - val_accuracy: 0.7987 - val_loss: 0.4717
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7997 - loss: 0.4187 - val_accuracy: 0.7727 - val_loss: 0.4700
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8029 - loss: 0.4108 - val_accuracy: 0.7922 - val_loss: 0.4710
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8127 - loss: 0.4022 - val_accuracy: 0.7922 - val_loss: 0.4667
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8208 - loss: 0.3957 - val_accuracy: 0.

In [62]:
def build_model(hp):
    model = Sequential()

    # Number of hidden layers
    num_layers = hp.Int('num_layers', min_value=1, max_value=5)

    for i in range(num_layers):
        units = hp.Int(f'units_{i}', min_value=8, max_value=128, step=8)
        activation = hp.Choice(f'activation_{i}', values=['relu', 'tanh', 'sigmoid'])

        if i == 0:
            model.add(Dense(units=units, activation=activation, input_dim=8))
        else:
            model.add(Dense(units=units, activation=activation))

    # Output layer
    model.add(Dense(1, activation="sigmoid"))

    # Compile
    model.compile(
        optimizer=hp.Choice('optimizer', values=['rmsprop', 'adam', 'sgd', 'nadam', 'adadelta']),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model



In [63]:
tuner = kt.RandomSearch(build_model,
                        objective = 'val_accuracy',
                        max_trials = 3,
                        directory = 'mydir',
                        project_name = 'final'
                        )

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [64]:
tuner.search(x_train,y_train,epochs = 5,validation_data = (x_test,y_test))

Trial 3 Complete [00h 00m 03s]
val_accuracy: 0.3571428656578064

Best val_accuracy So Far: 0.649350643157959
Total elapsed time: 00h 00m 08s


In [65]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 3,
 'units_0': 24,
 'activation_0': 'relu',
 'optimizer': 'sgd',
 'units_1': 8,
 'activation_1': 'relu',
 'units_2': 8,
 'activation_2': 'relu'}

In [66]:
model=tuner.get_best_models(num_models = 1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [68]:
model.fit(x_train , y_train,epochs = 200,initial_epoch=6,validation_data = (x_test,y_test))

Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.6759 - loss: 0.6575 - val_accuracy: 0.6494 - val_loss: 0.6620
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6775 - loss: 0.6522 - val_accuracy: 0.6494 - val_loss: 0.6566
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6775 - loss: 0.6465 - val_accuracy: 0.6429 - val_loss: 0.6513
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6726 - loss: 0.6411 - val_accuracy: 0.6364 - val_loss: 0.6467
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6661 - loss: 0.6363 - val_accuracy: 0.6429 - val_loss: 0.6423
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6629 - loss: 0.6317 - val_accuracy: 0.6494 - val_loss: 0.6382
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6612 - loss: 0.6275 - val_accuracy: 0.6494 - val_loss: 0.6339
Epoch 14/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6629 - loss: 0.6230 - val_accuracy: 0.64